# Week 2: Data Integration and Visualization with pandas

Real analysis usually requires more than one table. This lesson develops a reliable workflow for combining related datasets, checking whether joins behaved as expected, reshaping data, and building readable visualizations.

## Learning objectives

By the end of this lesson, you should be able to:

- explain the difference between `concat()`, `merge()`, and `join()`;
- identify join keys and check their uniqueness;
- validate a many-to-one merge and locate unmatched records;
- aggregate and reshape tidy data with `groupby()` and `pivot()`;
- create purposeful line and bar charts with Matplotlib;
- export an integrated dataset and a figure reproducibly.

> **Dataset note:** The included CSV files are small, fixed teaching snapshots inspired by public-health reporting. They are not current and must not be used for medical or policy decisions.


## About the teaching data

This notebook uses four small CSV files so that the integration steps are easy to inspect:

| File | Contents | Rows |
|---|---|---:|
| `covid_state_daily_part1.csv` | Daily cases and tests for March 1–5, 2021 | 20 |
| `covid_state_daily_part2.csv` | Daily cases and tests for March 6–10, 2021 | 20 |
| `state_metadata.csv` | State names, regions, and population in millions | 5 |
| `state_benchmarks.csv` | One teaching benchmark per analyzed state | 4 |

The two daily files contain four states—NSW, VIC, QLD, and WA—and use these columns:

- `date`: reporting date;
- `state`: abbreviated state key;
- `new_cases`: newly reported cases for that date;
- `new_tests`: tests reported for that date.

The data are fixed, simplified teaching snapshots designed for practicing pandas. They are **not live epidemiological data**, and the benchmark values are instructional rather than official public-health targets.




## 1. Setup and data sources

The notebook uses local files from the `data` folder. Fixed snapshots make the lesson reproducible: everyone sees the same rows and the notebook continues to work when an external website changes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 12)

DATA_DIR = Path("../data/data_raw")
PART1_PATH = DATA_DIR / "covid_state_daily_part1.csv"
PART2_PATH = DATA_DIR / "covid_state_daily_part2.csv"
METADATA_PATH = DATA_DIR / "state_metadata.csv"
BENCHMARK_PATH = DATA_DIR / "state_benchmarks.csv"

required_files = [PART1_PATH, PART2_PATH, METADATA_PATH, BENCHMARK_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing Week 2 data files. Keep the notebook beside the data folder: "
        + ", ".join(missing_files)
    )

print("pandas version:", pd.__version__)


## 2. Load and inspect each table

Before combining tables, establish their **grain**—what one row represents—and inspect their join keys.

| Table | One row represents | Expected key |
|---|---|---|
| daily parts | one state on one date | `date + state` |
| state metadata | one state | `state` |
| benchmarks | one state | `state` |


In [ ]:
part1 = pd.read_csv(PART1_PATH, parse_dates=["date"])
part2 = pd.read_csv(PART2_PATH, parse_dates=["date"])
state_metadata = pd.read_csv(METADATA_PATH)
state_benchmarks = pd.read_csv(BENCHMARK_PATH)

for name, frame in {
    "part1": part1,
    "part2": part2,
    "state_metadata": state_metadata,
    "state_benchmarks": state_benchmarks,
}.items():
    print(f"{name:17} shape={frame.shape}")


In [ ]:
display(part1.head())
display(state_metadata)
display(state_benchmarks)


In [ ]:
part1.info()


### Check keys before integrating

A key should match the grain of its table. Duplicate keys can multiply rows during a merge, producing plausible-looking but incorrect results.


In [ ]:
daily_key = ["date", "state"]

print("Part 1 duplicate keys:", int(part1.duplicated(daily_key).sum()))
print("Part 2 duplicate keys:", int(part2.duplicated(daily_key).sum()))
print("Metadata state key unique:", state_metadata["state"].is_unique)
print("Benchmark state key unique:", state_benchmarks["state"].is_unique)


## 3. Concatenate compatible tables

Use `pd.concat(..., axis=0)` when tables contain the same variables but different observations. `ignore_index=True` creates a clean sequential index; it does not remove duplicates.


In [ ]:
daily_cases = (
    pd.concat([part1, part2], axis=0, ignore_index=True)
    .sort_values(["date", "state"])
    .reset_index(drop=True)
)

print("Expected rows:", len(part1) + len(part2))
print("Combined rows:", len(daily_cases))
print("Duplicate date-state keys:", int(daily_cases.duplicated(daily_key).sum()))
daily_cases.head(8)


### Row-wise versus column-wise concatenation

- `axis=0` stacks rows and aligns columns by name.
- `axis=1` places objects side by side and aligns them by index.

Column-wise concatenation is safe only when the indexes have the intended meaning and alignment.


In [ ]:
left_example = pd.DataFrame({"state": ["NSW", "VIC"]}, index=[10, 20])
right_example = pd.DataFrame({"region": ["East", "South"]}, index=[10, 20])

pd.concat([left_example, right_example], axis=1)


## 4. Merge on explicit keys

`merge()` performs database-style joins. Choose the join type according to the analytical question.

| Join | Keeps |
|---|---|
| `inner` | keys present in both tables |
| `left` | every row from the left table |
| `right` | every row from the right table |
| `outer` | every key from either table |

For this lesson, every daily row should receive one metadata match, so the relationship is **many daily rows to one metadata row**.


In [ ]:
integrated = daily_cases.merge(
    state_metadata,
    how="left",
    on="state",
    validate="many_to_one",
    indicator=True,
)

integrated["_merge"].value_counts(dropna=False)


In [ ]:
unmatched_daily = integrated.loc[
    integrated["_merge"] != "both",
    ["date", "state", "_merge"],
]

print("Unmatched daily rows:", len(unmatched_daily))
integrated = integrated.drop(columns="_merge")
integrated.head()


### Diagnose keys with an outer merge

The metadata intentionally contains one state that has no rows in these daily snapshots. An outer merge plus `indicator=True` makes that difference visible instead of silently discarding it.


In [ ]:
key_audit = (
    daily_cases[["state"]].drop_duplicates()
    .merge(
        state_metadata[["state"]],
        on="state",
        how="outer",
        indicator=True,
    )
    .sort_values("state")
)

key_audit


## 5. Join an indexed table and derive measures

`DataFrame.join()` is convenient when the right-hand table is indexed by the key. Here we add one benchmark per state and calculate rates.

Because population is measured in millions, `new_cases / population_millions` directly gives daily cases per million people.


In [ ]:
benchmark_indexed = state_benchmarks.set_index("state")

integrated = integrated.join(
    benchmark_indexed,
    on="state",
    validate="many_to_one",
)

integrated["cases_per_million"] = (
    integrated["new_cases"] / integrated["population_millions"]
).round(2)
integrated["positivity_percent"] = (
    integrated["new_cases"] / integrated["new_tests"] * 100
).round(3)
integrated["above_target"] = (
    integrated["cases_per_million"] > integrated["target_cases_per_million"]
)

integrated.head()


> **Interpret carefully:** A normalized rate supports fairer comparisons across differently sized states, but it does not remove reporting differences, testing differences, or the instability of small counts.


## 6. Aggregate and reshape

Keep data in **tidy form** for most transformations: one observation per row and one variable per column. Aggregate with `groupby()` and reshape only when a report or visualization needs a different layout.


In [ ]:
state_summary = (
    integrated.groupby(["state", "state_name", "region"], as_index=False)
    .agg(
        total_cases=("new_cases", "sum"),
        average_daily_tests=("new_tests", "mean"),
        peak_daily_cases=("new_cases", "max"),
        average_cases_per_million=("cases_per_million", "mean"),
    )
    .sort_values("total_cases", ascending=False)
)

state_summary


In [ ]:
daily_cases_wide = integrated.pivot(
    index="date",
    columns="state",
    values="new_cases",
)

daily_cases_wide


`pivot()` requires each index-column pair to be unique. If duplicate pairs are legitimate and need aggregation, use `pivot_table(..., aggfunc=...)` instead.


In [ ]:
region_summary = (
    integrated.groupby("region", as_index=False)
    .agg(total_cases=("new_cases", "sum"), total_tests=("new_tests", "sum"))
)
region_summary["positivity_percent"] = (
    region_summary["total_cases"] / region_summary["total_tests"] * 100
).round(3)
region_summary


## 7. Visualize with a question in mind

Before plotting, state the question:

- **How did daily cases change over time?** Use a line chart.
- **Which states recorded the most cases in the snapshot?** Use a sorted bar chart.

Good charts include a descriptive title, labeled axes, readable scales, and a legend only when it adds information.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

daily_cases_wide.plot(
    ax=axes[0],
    marker="o",
    linewidth=2,
)
axes[0].set(
    title="Daily reported cases by state",
    xlabel="Date",
    ylabel="New cases",
)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(title="State", frameon=False)

bar_data = state_summary.sort_values("total_cases")
axes[1].barh(bar_data["state"], bar_data["total_cases"], color="#4C78A8")
axes[1].set(
    title="Total cases in the 10-day snapshot",
    xlabel="Cases",
    ylabel="State",
)
axes[1].grid(axis="x", alpha=0.25)

plt.show()


### A heatmap for comparison

A heatmap can reveal relative highs and lows across two dimensions. It should complement—not replace—the underlying numeric table.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
image = ax.imshow(daily_cases_wide.T, aspect="auto", cmap="Blues")

ax.set_yticks(range(len(daily_cases_wide.columns)), labels=daily_cases_wide.columns)
ax.set_xticks(
    range(len(daily_cases_wide.index)),
    labels=daily_cases_wide.index.strftime("%b %d"),
    rotation=45,
    ha="right",
)
ax.set(title="Daily cases heatmap", xlabel="Date", ylabel="State")
fig.colorbar(image, ax=ax, label="New cases")
fig.tight_layout()
plt.show()


### Why not default to a pie chart?

Pie charts make close values hard to compare and become crowded quickly. A sorted bar chart generally communicates magnitude more accurately. If a pie chart is used, values should be calculated from the DataFrame—not typed manually—and the number of slices should be small.


## 8. Validate and export

Validation turns assumptions into executable checks. Export only after the integrated result passes them.


In [ ]:
assert len(daily_cases) == len(part1) + len(part2)
assert not daily_cases.duplicated(["date", "state"]).any()
assert integrated[["state_name", "population_millions"]].notna().all().all()
assert (integrated["new_cases"] >= 0).all()
assert (integrated["new_tests"] > 0).all()

export_path = Path("week2_integrated_cases.csv")
integrated.to_csv(export_path, index=False)

print("Validation passed.")
print("Exported:", export_path.resolve())


## 9. A reusable integration function

Packaging the workflow in a function reduces copy-and-paste errors and makes the assumptions testable.


In [ ]:
def integrate_case_data(parts, metadata, benchmarks):
    'Combine daily parts, attach state information, and calculate rates.'
    combined = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["date", "state"])
        .reset_index(drop=True)
    )

    if combined.duplicated(["date", "state"]).any():
        raise ValueError("Duplicate date-state keys found")

    result = combined.merge(
        metadata,
        on="state",
        how="left",
        validate="many_to_one",
    ).join(
        benchmarks.set_index("state"),
        on="state",
        validate="many_to_one",
    )

    if result["population_millions"].isna().any():
        raise ValueError("At least one state did not match the metadata table")

    result["cases_per_million"] = (
        result["new_cases"] / result["population_millions"]
    ).round(2)
    return result


In [ ]:
integrated_again = integrate_case_data(
    [part1, part2],
    state_metadata,
    state_benchmarks,
)

print("Rows returned:", len(integrated_again))
integrated_again.head()


## Key takeaways

- Inspect the grain and keys of every table before combining it.
- Use `concat()` to stack compatible observations and `merge()` for key-based relationships.
- Use `validate=` and `indicator=True` to expose incorrect assumptions.
- Aggregate in tidy form; pivot only when a wide layout is useful.
- Build charts around a question and calculate every plotted value from the data.
- Validate before exporting and preserve the original inputs.

The separate **Week 2 Checkpoint Assignment** contains the graded end-of-session tasks.
